In [1]:
import duckdb
import ollama
import re
import time

# Setup
DB_PATH = "../data/omop_clinical.duckdb"
MODEL_NAME = "qwen2.5-coder:7b"
SIMILARITY_THRESHOLD = 0.90 # 90% confidence minimum to accept a match

def get_unique_unmapped_conditions():
    """Fetch ALL unique unmapped clinical conditions from DuckDB."""
    print("🔌 Connecting to DuckDB to fetch unmapped conditions...")
    with duckdb.connect(DB_PATH) as con:
        query = """
            SELECT DISTINCT condition_source_value
            FROM condition_occurrence
            WHERE condition_concept_id = 0
            AND condition_source_value IS NOT NULL
        """
        return [row[0] for row in con.execute(query).fetchall()]

def ai_semantic_normalization(raw_term):
    """Uses local LLM to normalize a messy clinical term into a core standard name."""
    system_prompt = """
    You are an expert Clinical Data Informatician.
    Normalize raw, messy clinical text into a clean, core medical term.
    RULES:
    1. Respond ONLY with the core medical term.
    2. Do NOT include any numeric IDs.
    3. Do NOT include any tags in parentheses like '(disorder)', '(finding)', or '(person)'.
    4. Keep it as short and precise as possible.
    """
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f"Raw clinical text: '{raw_term}'"}
            ]
        )
        clean_text = response['message']['content'].strip().strip("'").strip('"')
        clean_text = re.sub(r'\([^)]*\)', '', clean_text).strip()
        return clean_text
    except Exception as e:
        return None

def find_best_match(con, normalized_term):
    """Finds the best OMOP concept using Jaro-Winkler similarity."""
    # We calculate similarity across the concept dictionary and order by the highest score
    search_query = """
        SELECT concept_id, concept_name, domain_id, 
               jaro_winkler_similarity(LOWER(concept_name), LOWER(?)) AS score
        FROM concept 
        WHERE vocabulary_id = 'SNOMED'
        ORDER BY score DESC
        LIMIT 1
    """
    match = con.execute(search_query, [normalized_term]).fetchone()
    
    # Only return the match if the AI's translation is >= 90% similar to the DB Dictionary
    if match and match[3] >= SIMILARITY_THRESHOLD:
        return match
    return None

def update_database(con, raw_term, concept_id):
    """Updates ALL occurrences of the raw term with the newly found concept_id."""
    update_query = """
        UPDATE condition_occurrence
        SET condition_concept_id = ?
        WHERE condition_source_value = ? 
          AND condition_concept_id = 0
    """
    con.execute(update_query, [concept_id, raw_term])

# EXECUTION BLOCK
print("⚙️ STARTING AI-ASSISTED SEMANTIC MAPPING (V3: PRODUCTION READY)\n" + "-"*50)

unique_terms = get_unique_unmapped_conditions()

if not unique_terms:
    print("✅ No unmapped conditions found! Database is fully normalized.")
else:
    total_terms = len(unique_terms)
    print(f"⚠️ Found {total_terms} UNIQUE unmapped terms. Starting AI pipeline...\n")
    
    success_count = 0
    
    with duckdb.connect(DB_PATH) as con:
        for i, term in enumerate(unique_terms, 1):
            print(f"[{i}/{total_terms}] Raw: '{term}'")
            
            # 1. AI Normalization
            ai_term = ai_semantic_normalization(term)
            if not ai_term:
                print("   ❌ AI Failed to respond.")
                continue
            
            print(f"   ✨ AI:  '{ai_term}'")
            
            # 2. Database Fuzzy Match
            start_time = time.time()
            match = find_best_match(con, ai_term)
            db_time = time.time() - start_time
            
            # 3. Validation & Write-back
            if match:
                concept_id, concept_name, domain, score = match
                print(f"   🎯 DB:  '{concept_name}' (ID: {concept_id}) | Score: {score:.2f} | Time: {db_time:.1f}s")
                
                # The crucial step: writing the fix back to the actual database
                update_database(con, term, concept_id)
                success_count += 1
            else:
                print(f"   ❌ DB:  No match found above {SIMILARITY_THRESHOLD*100}% confidence.")
            print("-" * 40)
            
    print("\n✅ AI MAPPING COMPLETE!")
    print(f"📊 Successfully mapped and updated {success_count} out of {total_terms} unique terms.")

⚙️ STARTING AI-ASSISTED SEMANTIC MAPPING (V3: PRODUCTION READY)
--------------------------------------------------
🔌 Connecting to DuckDB to fetch unmapped conditions...
⚠️ Found 39 UNIQUE unmapped terms. Starting AI pipeline...

[1/39] Raw: 'Refugee (person)'
   ✨ AI:  'refugee'
   🎯 DB:  'Refugee' (ID: 40562294) | Score: 1.00 | Time: 0.5s
----------------------------------------
[2/39] Raw: 'History of seizure (situation)'
   ✨ AI:  'Seizure'
   🎯 DB:  'Seizure' (ID: 377091) | Score: 1.00 | Time: 0.1s
----------------------------------------
[3/39] Raw: 'Unhealthy alcohol drinking behavior (finding)'
   ✨ AI:  'Alcohol abuse'
   🎯 DB:  'Alcohol oxidase' (ID: 4305340) | Score: 0.92 | Time: 0.1s
----------------------------------------
[4/39] Raw: 'Fractured dental filling (finding)'
   ✨ AI:  'Dental filling fracture'
   🎯 DB:  'Dental filling surface rough' (ID: 4150430) | Score: 0.92 | Time: 0.1s
----------------------------------------
[5/39] Raw: 'Has a criminal record (finding)'
